# Test

In [21]:
import torch
import torch.nn as nn
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel
from architecture import VisionConnector, SimpleVLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------0) LOAD MODELS (FROZEN) ---------
LLM_NAME = "Qwen/Qwen2.5-0.5B"
VISION_NAME = "google/siglip-base-patch16-224"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, dtype=torch.float16
).to(DEVICE)
llm.requires_grad_(False)
llm.eval()

vision_processor = AutoProcessor.from_pretrained(VISION_NAME, use_fast=True)
vision_model = AutoModel.from_pretrained(
    VISION_NAME, dtype=torch.float16
).to(DEVICE)
vision_model.requires_grad_(False)
vision_model.eval()

# ---------1) ADD IMAGE TOKEN ---------
IMAGE_TOKEN = "<image>"
if IMAGE_TOKEN not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"additional_special_tokens": [IMAGE_TOKEN]})
    llm.resize_token_embeddings(len(tokenizer))

# ---------2) CONNECTOR (SHALLOW MLP) ---------
# SigLIP output dimension
vision_dim = vision_model.config.vision_config.hidden_size
llm_dim = llm.config.hidden_size

connector = VisionConnector(
    vision_dim=vision_dim,
    llm_dim=llm_dim,
    hidden_dim=4096
).to(DEVICE).half()

vlm = SimpleVLM(
    vision_encoder=vision_model.vision_model,
    llm=llm,
    connector=connector
).to(DEVICE)

optimizer = torch.optim.AdamW(connector.parameters(), lr=1e-3)

# ---------3) TRAIN STEP (ALIGNMENT) ---------
img = Image.open("example.jpg").convert("RGB")
caption = "A dog running in a grassy field."

labels = tokenizer(caption, return_tensors="pt").input_ids.to(DEVICE)
inputs_embeds = vision_processor(images=img, return_tensors="pt").to(DEVICE)


text_input = f"{IMAGE_TOKEN} {caption}"
token = tokenizer(text_input, return_tensors="pt")
input_ids = token.input_ids.to(DEVICE)
attention_mask = token.attention_mask.to(DEVICE)
image_token_id = tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
idx = (input_ids == image_token_id).nonzero()[0, 1]

with torch.no_grad():
    feats = vision_model.vision_model(inputs_embeds["pixel_values"]).last_hidden_state
    K = feats.size(1)

labels = torch.cat([
    labels[:, :idx],  # text before image
    torch.full((1, K), -100, device=DEVICE, dtype=labels.dtype),  # ignore image tokens
    labels[:, idx:]  # text after image
], dim=1)

out = vlm(images=inputs_embeds["pixel_values"], input_ids=input_ids, image_token_id=image_token_id, labels=labels)
loss = out.loss

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

# ---------6) INFERENCE TEST ---------
with torch.no_grad():
    prompt = "What is in this image?"
    generated = llm.generate(input_ids=input_ids, max_new_tokens=40, attention_mask=attention_mask, pad_token_id=tokenizer.pad_token_id)

print(tokenizer.decode(generated[0], skip_special_tokens=True))

Loss: 4.693543434143066
 A dog running in a grassy field. A dog running in a grassy field. A dog running in a grassy field. A dog running in a grassy field. A dog running in a grassy field. A dog running in


In [9]:
llm.config

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "rms_n